In [8]:
import pathlib
from typing import Dict

import numpy as np
from rich.pretty import pprint

from oqd_dataschema import (
    CastDataset,
    Dataset,
    Datastore,
    GroupBase,
    GroupRegistry,
    condataset,
    Attrs,
    CastFolder,
    CastTable,
)


In [9]:
from oqd_dataschema.folder import Folder
from pydantic import Field

class DatastoreGroup(GroupBase):
    """
    Here we define a custom Group, which is automatically added at runtime to the GroupRegistry.
    """
    attrs: Attrs = Field(
        default_factory=lambda: dict(
            timestamp=str(datetime.datetime.now(datetime.timezone.utc))
        )
    )
    document: Dict[str, CastDataset]


/Users/vishnu/Documents/OQD/oqd-analog-emulator/.venv/lib/python3.11/site-packages/oqd_dataschema/group.py:119: UserWarning: Overwriting previously registered `DatastoreGroup` group of the same name.
  GroupRegistry.register(cls)


In [10]:
GroupRegistry.groups

{'DatastoreGroup': __main__.DatastoreGroup,
 'RegisterGroup': oqd_analog_emulator.data.RegisterGroup,
 'MetricGroup': oqd_analog_emulator.data.MetricGroup}

In [11]:
from datetime import datetime

dtype = "int32"
data = np.ones([10, 10]).astype(dtype)
custom_group = DatastoreGroup(
    document=dict(
        a=Dataset(
            data=np.array([1]),
        ),
        b=Dataset(
            data=np.array([np.arange(10).tobytes()]),
        ),
    ),
    attrs={"created_at": f"{datetime.now()}"},
)


pprint(custom_group)

store = Datastore()

store.add(group1=custom_group)

store.model_dump_hdf5(pathlib.Path("test.h5"))

DatastoreGroup(
│   attrs={'created_at': '2026-11-28 01:45:32.205153'},
│   document={
│   │   'a': Dataset(attrs={}, dtype='int64', shape=(1,), data=array([1])),
│   │   'b': Dataset(
│   │   │   attrs={},
│   │   │   dtype='bytes',
│   │   │   shape=(1,),
│   │   │   data=array([b'\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\x00\x00\x00\x00\x03\x00\x00\x00\x00\x00\x00\x00\x04\x00\x00\x00\x00\x00\x00\x00\x05\x00\x00\x00\x00\x00\x00\x00\x06\x00\x00\x00\x00\x00\x00\x00\x07\x00\x00\x00\x00\x00\x00\x00\x08\x00\x00\x00\x00\x00\x00\x00\t'],
│     dtype='|S80')
│   │   )
│   },
│   class_='DatastoreGroup'
)

In [12]:
np.frombuffer(np.arange(10).tobytes(), dtype="int")

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [13]:
from oqd_analog_emulator.qutip import QutipBackend, QutipMethodTableOptions
backend = QutipBackend()
options = QutipMethodTableOptions(
    metrics=dict(
        X_q0=("%X", "q[0]"),
    ),
    # solver_options = dict(progress_bar="enhanced"),
)
program = """
q = qreg(2)
initialize(q)

evolve(%X, 1, q[0])
evolve(%X, 1, q[1])
a = measure(q)

q
"""
out, _, cfg, interp = backend.run(program, options=options)
interp.get_data("test")